# Επαναλαμβανόμενα νευρωνικά δίκτυα

Στο προηγούμενο μάθημα, καλύψαμε πλούσιες σημασιολογικές αναπαραστάσεις κειμένου. Η αρχιτεκτονική που χρησιμοποιούμε αποτυπώνει τη συνολική σημασία των λέξεων σε μια πρόταση, αλλά δεν λαμβάνει υπόψη τη **σειρά** των λέξεων, επειδή η λειτουργία συσσώρευσης που ακολουθεί τα ενσωματώματα αφαιρεί αυτή την πληροφορία από το αρχικό κείμενο. Επειδή αυτά τα μοντέλα δεν μπορούν να αναπαραστήσουν τη σειρά των λέξεων, δεν μπορούν να λύσουν πιο σύνθετα ή ασαφή καθήκοντα όπως η δημιουργία κειμένου ή η απάντηση σε ερωτήσεις.

Για να αποτυπώσουμε το νόημα μιας ακολουθίας κειμένου, θα χρησιμοποιήσουμε μια αρχιτεκτονική νευρωνικού δικτύου που ονομάζεται **επαναλαμβανόμενο νευρωνικό δίκτυο**, ή RNN. Όταν χρησιμοποιούμε RNN, περνάμε την πρότασή μας μέσα από το δίκτυο ένα token κάθε φορά, και το δίκτυο παράγει κάποια **κατάσταση**, την οποία στη συνέχεια δίνουμε ξανά στο δίκτυο μαζί με το επόμενο token.

![Εικόνα που δείχνει ένα παράδειγμα δημιουργίας επαναλαμβανόμενου νευρωνικού δικτύου.](../../../../../translated_images/el/rnn.27f5c29c53d727b5.webp)

Δεδομένης της ακολουθίας εισόδου token $X_0,\dots,X_n$, το RNN δημιουργεί μια ακολουθία από μπλοκ νευρωνικών δικτύων, και εκπαιδεύει αυτή την ακολουθία με end-to-end εκπαίδευση χρησιμοποιώντας επιστροφή λαθών (backpropagation). Κάθε μπλοκ δικτύου δέχεται ως είσοδο το ζεύγος $(X_i,S_i)$, και παράγει ως αποτέλεσμα το $S_{i+1}$. Η τελική κατάσταση $S_n$ ή η έξοδος $Y_n$ εισάγεται σε έναν γραμμικό ταξινομητή για να παράγει το αποτέλεσμα. Όλα τα μπλοκ δικτύου μοιράζονται τα ίδια βάρη, και εκπαιδεύονται end-to-end μέσω μίας διαδρομής επιστροφής λαθών.

> Η παραπάνω εικόνα δείχνει το επαναλαμβανόμενο νευρωνικό δίκτυο σε ξεδιπλωμένη μορφή (αριστερά), και σε πιο συμπαγή επαναλαμβανόμενη αναπαράσταση (δεξιά). Είναι σημαντικό να κατανοήσουμε ότι όλα τα κελιά RNN έχουν τα ίδια **κοινόχρηστα βάρη**.

Επειδή οι διανύσματα κατάστασης $S_0,\dots,S_n$ περνούν μέσα από το δίκτυο, το RNN μπορεί να μάθει αλληλουχίες εξαρτήσεων μεταξύ των λέξεων. Για παράδειγμα, όταν η λέξη *όχι* εμφανίζεται κάπου στην ακολουθία, μπορεί να μάθει να αρνείται ορισμένα στοιχεία μέσα στο διάνυσμα κατάστασης.

Εσωτερικά, κάθε κελί RNN περιέχει δύο πίνακες βαρών: $W_H$ και $W_I$, και μια παράμετρο εκκενώσεως $b$. Σε κάθε βήμα RNN, δεδομένης της εισόδου $X_i$ και της εισόδου κατάστασης $S_i$, η έξοδος κατάστασης υπολογίζεται ως $S_{i+1} = f(W_H\times S_i + W_I\times X_i+b)$, όπου $f$ είναι μια συνάρτηση ενεργοποίησης (συχνά $\tanh$).

> Για προβλήματα όπως η δημιουργία κειμένου (που θα καλύψουμε στην επόμενη ενότητα) ή η μηχανική μετάφραση, θέλουμε επίσης να λάβουμε μια τιμή εξόδου σε κάθε βήμα RNN. Σε αυτή την περίπτωση, υπάρχει επίσης ένας άλλος πίνακας $W_O$, και η έξοδος υπολογίζεται ως $Y_i=f(W_O\times S_i+b_O)$.

Ας δούμε πώς τα επαναλαμβανόμενα νευρωνικά δίκτυα μπορούν να μας βοηθήσουν να ταξινομήσουμε το σύνολο δεδομένων των ειδήσεών μας.

> Για το sandbox περιβάλλον, πρέπει να εκτελέσουμε το παρακάτω κελί για να βεβαιωθούμε ότι η απαιτούμενη βιβλιοθήκη είναι εγκατεστημένη και τα δεδομένα έχουν προ-φορτωθεί. Αν τρέχετε τοπικά, μπορείτε να παραλείψετε το παρακάτω κελί.


In [1]:
import sys
!{sys.executable} -m pip install --quiet tensorflow_datasets==4.4.0
!cd ~ && wget -q -O - https://mslearntensorflowlp.blob.core.windows.net/data/tfds-ag-news.tgz | tar xz

In [2]:
import tensorflow as tf
from tensorflow import keras
import tensorflow_datasets as tfds
import numpy as np

# We are going to be training pretty large models. In order not to face errors, we need
# to set tensorflow option to grow GPU memory allocation when required
physical_devices = tf.config.list_physical_devices('GPU') 
if len(physical_devices)>0:
    tf.config.experimental.set_memory_growth(physical_devices[0], True)

ds_train, ds_test = tfds.load('ag_news_subset').values()

Κατά την εκπαίδευση μεγάλων μοντέλων, η κατανομή μνήμης GPU μπορεί να γίνει πρόβλημα. Μπορεί επίσης να χρειαστεί να πειραματιστούμε με διαφορετικά μεγέθη minibatch, έτσι ώστε τα δεδομένα να χωρούν στη μνήμη της GPU μας, ενώ η εκπαίδευση να είναι αρκετά γρήγορη. Εάν τρέχετε αυτόν τον κώδικα στη δική σας GPU, μπορείτε να πειραματιστείτε προσαρμόζοντας το μέγεθος minibatch για να επιταχύνετε την εκπαίδευση.

> **Note**: Ορισμένες εκδόσεις οδηγών NVidia είναι γνωστό ότι δεν απελευθερώνουν τη μνήμη μετά την εκπαίδευση του μοντέλου. Τρέχουμε αρκετά παραδείγματα σε αυτό το σημειωματάριο, και αυτό μπορεί να προκαλέσει εξάντληση μνήμης σε ορισμένες ρυθμίσεις, ειδικά αν κάνετε τους δικούς σας πειραματισμούς ως μέρος του ίδιου σημειωματαρίου. Εάν αντιμετωπίσετε περίεργα σφάλματα κατά την έναρξη της εκπαίδευσης του μοντέλου, ίσως θέλετε να επανεκκινήσετε τον πυρήνα του σημειωματαρίου.


In [3]:
batch_size = 16
embed_size = 64

## Απλός ταξινομητής RNN

Στην περίπτωση ενός απλού RNN, κάθε επαναλαμβανόμενη μονάδα είναι ένα απλό γραμμικό δίκτυο, το οποίο λαμβάνει ένα διάνυσμα εισόδου και ένα διάνυσμα κατάστασης, και παράγει ένα νέο διάνυσμα κατάστασης. Στο Keras, αυτό μπορεί να αντιπροσωπευτεί από το επίπεδο `SimpleRNN`.

Ενώ μπορούμε να δώσουμε απευθείας στην RNN στρώση διανύσματα κωδικοποιημένα με one-hot, αυτό δεν είναι καλή ιδέα λόγω του υψηλού τους διαστατικού εύρους. Επομένως, θα χρησιμοποιήσουμε μία στρώση ενσωμάτωσης για να μειώσουμε τη διάσταση των διανυσμάτων λέξεων, ακολουθούμενη από μια στρώση RNN, και τελικά έναν ταξινομητή `Dense`.

> **Σημείωση**: Σε περιπτώσεις όπου η διάσταση δεν είναι τόσο υψηλή, για παράδειγμα κατά τη χρήση χαρακτήρων ως tokens, μπορεί να έχει νόημα να δοθούν απευθείας τα one-hot κωδικοποιημένα tokens μέσα στο κελί RNN.


In [4]:
vocab_size = 20000

vectorizer = keras.layers.experimental.preprocessing.TextVectorization(
    max_tokens=vocab_size,
    input_shape=(1,))

model = keras.models.Sequential([
    vectorizer,
    keras.layers.Embedding(vocab_size, embed_size),
    keras.layers.SimpleRNN(16),
    keras.layers.Dense(4,activation='softmax')
])

model.summary()

Model: "sequential"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
text_vectorization (TextVect (None, None)              0         
_________________________________________________________________
embedding (Embedding)        (None, None, 64)          1280000   
_________________________________________________________________
simple_rnn (SimpleRNN)       (None, 16)                1296      
_________________________________________________________________
dense (Dense)                (None, 4)                 68        
Total params: 1,281,364
Trainable params: 1,281,364
Non-trainable params: 0
_________________________________________________________________


> **Σημείωση:** Χρησιμοποιούμε ένα μη εκπαιδευμένο επίπεδο ενσωμάτωσης εδώ για απλότητα, αλλά για καλύτερα αποτελέσματα μπορούμε να χρησιμοποιήσουμε ένα προεκπαιδευμένο επίπεδο ενσωμάτωσης χρησιμοποιώντας το Word2Vec, όπως περιγράφεται στην προηγούμενη μονάδα. Θα ήταν μια καλή άσκηση για εσάς να προσαρμόσετε αυτόν τον κώδικα ώστε να λειτουργεί με προεκπαιδευμένες ενσωματώσεις.

Τώρα ας εκπαιδεύσουμε το RNN μας. Τα RNN γενικά είναι αρκετά δύσκολα στην εκπαίδευση, επειδή μόλις οι RNN μονάδες ξεδιπλωθούν κατά το μήκος της αλληλουχίας, ο τελικός αριθμός των επιπέδων που εμπλέκονται στην οπισθοδιάδοση είναι αρκετά μεγάλος. Έτσι, πρέπει να επιλέξουμε μικρότερο ρυθμό μάθησης και να εκπαιδεύσουμε το δίκτυο σε ένα μεγαλύτερο σύνολο δεδομένων για να παραχθούν καλά αποτελέσματα. Αυτό μπορεί να πάρει αρκετό χρόνο, γι' αυτό η χρήση GPU προτιμάται.

Για να επιταχύνουμε τα πράγματα, θα εκπαιδεύσουμε το μοντέλο RNN μόνο στους τίτλους των ειδήσεων, παραλείποντας την περιγραφή. Μπορείτε να δοκιμάσετε να εκπαιδεύσετε με την περιγραφή και να δείτε αν μπορείτε να πετύχετε την εκπαίδευση του μοντέλου.


In [5]:
def extract_title(x):
    return x['title']

def tupelize_title(x):
    return (extract_title(x),x['label'])

print('Training vectorizer')
vectorizer.adapt(ds_train.take(2000).map(extract_title))

Training vectorizer


In [6]:
model.compile(loss='sparse_categorical_crossentropy',metrics=['acc'], optimizer='adam')
model.fit(ds_train.map(tupelize_title).batch(batch_size),validation_data=ds_test.map(tupelize_title).batch(batch_size))

7500/7500 [==============================] - 82s 11ms/step - loss: 0.6629 - acc: 0.7623 - val_loss: 0.5559 - val_acc: 0.7995


> **Σημείωση** ότι η ακρίβεια είναι πιθανόν να είναι χαμηλότερη εδώ, επειδή εκπαιδευόμαστε μόνο σε τίτλους ειδήσεων.


## Revisiting variable sequences 

Remember that the `TextVectorization` layer will automatically pad sequences of variable length in a minibatch with pad tokens. It turns out that those tokens also take part in training, and they can complicate convergence of the model.

There are several approaches we can take to minimize the amount of padding. One of them is to reorder the dataset by sequence length and group all sequences by size. This can be done using the `tf.data.experimental.bucket_by_sequence_length` function (see [documentation](https://www.tensorflow.org/api_docs/python/tf/data/experimental/bucket_by_sequence_length)). 

Another approach is to use **masking**. In Keras, some layers support additional input that shows which tokens should be taken into account when training. To incorporate masking into our model, we can either include a separate `Masking` layer ([docs](https://keras.io/api/layers/core_layers/masking/)), or we can specify the `mask_zero=True` parameter of our `Embedding` layer.

> **Note**: This training will take around 5 minutes to complete one epoch on the whole dataset. Feel free to interrupt training at any time if you run out of patience. What you can also do is limit the amount of data used for training, by adding `.take(...)` clause after `ds_train` and `ds_test` datasets.


In [7]:
def extract_text(x):
    return x['title']+' '+x['description']

def tupelize(x):
    return (extract_text(x),x['label'])

model = keras.models.Sequential([
    vectorizer,
    keras.layers.Embedding(vocab_size,embed_size,mask_zero=True),
    keras.layers.SimpleRNN(16),
    keras.layers.Dense(4,activation='softmax')
])

model.compile(loss='sparse_categorical_crossentropy',metrics=['acc'], optimizer='adam')
model.fit(ds_train.map(tupelize).batch(batch_size),validation_data=ds_test.map(tupelize).batch(batch_size))

7500/7500 [==============================] - 371s 49ms/step - loss: 0.5401 - acc: 0.8079 - val_loss: 0.3780 - val_acc: 0.8822


Τώρα που χρησιμοποιούμε μάσκες, μπορούμε να εκπαιδεύσουμε το μοντέλο σε ολόκληρο το σύνολο δεδομένων τίτλων και περιγραφών.

> **Σημείωση**: Έχετε παρατηρήσει ότι έχουμε χρησιμοποιήσει έναν διανυσματοποιητή εκπαιδευμένο στους τίτλους των ειδήσεων, και όχι σε ολόκληρο το κύριο σώμα του άρθρου; Ενδεχομένως, αυτό μπορεί να κάνει μερικά από τα tokens να αγνοηθούν, οπότε είναι καλύτερο να επαναεκπαιδεύσουμε τον διανυσματοποιητή. Ωστόσο, μπορεί να έχει πολύ μικρή επίδραση, οπότε θα επιμείνουμε στον προηγούμενο προ-εκπαιδευμένο διανυσματοποιητή για λόγους απλότητας.


## LSTM: Μακροπρόθεσμη βραχυπρόθεσμη μνήμη

Ένα από τα κύρια προβλήματα των RNN είναι οι **εξαφανιζόμενοι βαθμίδες**. Τα RNN μπορεί να είναι αρκετά μακριά και μπορεί να δυσκολεύονται να διασπείρουν τις βαθμίδες όλη την πορεία μέχρι το πρώτο επίπεδο του δικτύου κατά την αντίστροφη διάδοση. Όταν συμβαίνει αυτό, το δίκτυο δεν μπορεί να μάθει σχέσεις μεταξύ απομακρυσμένων στοιχείων. Ένας τρόπος να αποφύγουμε αυτό το πρόβλημα είναι να εισαγάγουμε **ρητή διαχείριση κατάστασης** χρησιμοποιώντας **πύλες**. Οι δύο πιο κοινές αρχιτεκτονικές που εισάγουν πύλες είναι η **μακροπρόθεσμη βραχυπρόθεσμη μνήμη** (LSTM) και η **μπλοκαρισμένη μονάδα αναμετάδοσης** (GRU). Εδώ θα καλύψουμε τις LSTM.

![Εικόνα που δείχνει ένα παράδειγμα κυττάρου μακροπρόθεσμης βραχυπρόθεσμης μνήμης](../../../../../lessons/5-NLP/16-RNN/images/long-short-term-memory-cell.svg)

Ένα δίκτυο LSTM οργανώνεται με παρόμοιο τρόπο με ένα RNN, αλλά υπάρχουν δύο καταστάσεις που διαβιβάζονται από επίπεδο σε επίπεδο: η πραγματική κατάσταση $c$, και ο κρυφός διάνυσμα $h$. Σε κάθε μονάδα, ο κρυφός διάνυσμα $h_{t-1}$ συνδυάζεται με την είσοδο $x_t$, και μαζί ελέγχουν τι συμβαίνει στην κατάσταση $c_t$ και στην έξοδο $h_{t}$ μέσω των **πυλών**. Κάθε πύλη έχει ενεργοποίηση sigmoid (έξοδος στο εύρος $[0,1]$), η οποία μπορεί να θεωρηθεί ως μάσκα bitwise όταν πολλαπλασιαστεί με το διάνυσμα κατάστασης. Οι LSTM έχουν τις εξής πύλες (από αριστερά προς τα δεξιά στην παραπάνω εικόνα):
* **πύλη λήθης** που καθορίζει ποια συστατικά του διανύσματος $c_{t-1}$ πρέπει να ξεχάσουμε και ποια να περάσουμε.
* **πύλη εισόδου** που καθορίζει πόση πληροφορία από το διάνυσμα εισόδου και το προηγούμενο κρυφό διάνυσμα πρέπει να ενσωματωθεί στο διάνυσμα κατάστασης.
* **πύλη εξόδου** που παίρνει το νέο διάνυσμα κατάστασης και αποφασίζει ποια από τα συστατικά του θα χρησιμοποιηθούν για να παραχθεί το νέο κρυφό διάνυσμα $h_t$.

Τα συστατικά της κατάστασης $c$ μπορούν να θεωρηθούν ως σημαίες που μπορούν να ενεργοποιηθούν και να απενεργοποιηθούν. Για παράδειγμα, όταν συναντούμε το όνομα *Alice* στην ακολουθία, υποθέτουμε ότι αναφέρεται σε γυναίκα, και σηκώνουμε τη σημαία στην κατάσταση που λέει ότι έχουμε θηλυκό ουσιαστικό στην πρόταση. Όταν συναντήσουμε περαιτέρω τις λέξεις *and Tom*, θα ανεβάσουμε τη σημαία που λέει ότι έχουμε πληθυντικό ουσιαστικό. Έτσι, με τη διαχείριση της κατάστασης μπορούμε να παρακολουθούμε τις γραμματικές ιδιότητες της πρότασης.

> **Σημείωση**: Εδώ είναι μια εξαιρετική πηγή για να κατανοήσουμε τις εσωτερικές λειτουργίες των LSTM: [Understanding LSTM Networks](https://colah.github.io/posts/2015-08-Understanding-LSTMs/) του Christopher Olah.

Ενώ η εσωτερική δομή ενός κυττάρου LSTM μπορεί να φαίνεται πολύπλοκη, το Keras κρύβει αυτή την υλοποίηση μέσα στο επίπεδο `LSTM`, οπότε το μόνο που χρειάζεται να κάνουμε στο παραπάνω παράδειγμα είναι να αντικαταστήσουμε το επαναλαμβανόμενο επίπεδο:


In [8]:
model = keras.models.Sequential([
    vectorizer,
    keras.layers.Embedding(vocab_size, embed_size),
    keras.layers.LSTM(8),
    keras.layers.Dense(4,activation='softmax')
])

model.compile(loss='sparse_categorical_crossentropy',metrics=['acc'], optimizer='adam')
model.fit(ds_train.map(tupelize).batch(8),validation_data=ds_test.map(tupelize).batch(8))

15000/15000 [==============================] - 188s 13ms/step - loss: 0.5692 - acc: 0.7916 - val_loss: 0.3441 - val_acc: 0.8870


> **Σημείωση** ότι η εκπαίδευση των LSTM είναι επίσης αρκετά αργή, και μπορεί να μην δείτε μεγάλη αύξηση στην ακρίβεια στην αρχή της εκπαίδευσης. Μπορεί να χρειαστεί να συνεχίσετε την εκπαίδευση για κάποιο χρονικό διάστημα για να επιτύχετε καλή ακρίβεια.


## Αμφίδρομα και πολυεπίπεδα RNNs

Στα παραδείγματα μας μέχρι τώρα, τα επαναληπτικά δίκτυα λειτουργούν από την αρχή μιας ακολουθίας μέχρι το τέλος. Αυτό μας φαίνεται φυσικό γιατί ακολουθεί την ίδια κατεύθυνση με την οποία διαβάζουμε ή ακούμε ομιλία. Ωστόσο, για σενάρια που απαιτούν τυχαία πρόσβαση στην ακολουθία εισόδου, έχει περισσότερο νόημα να εκτελείται ο επαναληπτικός υπολογισμός και στις δύο κατευθύνσεις. Τα RNNs που επιτρέπουν υπολογισμούς και στις δύο κατευθύνσεις ονομάζονται **αμφίδρομα** RNNs, και μπορούν να δημιουργηθούν τυλίγοντας το επαναληπτικό επίπεδο με ένα ειδικό επίπεδο `Bidirectonal`.

> **Σημείωση**: Το επίπεδο `Bidirectional` φτιάχνει δύο αντίγραφα του επίπεδου μέσα σε αυτό, και ορίζει την ιδιότητα `go_backwards` ενός από αυτά τα αντίγραφα σε `True`, κάνοντάς το να κινείται στην αντίθετη κατεύθυνση κατά μήκος της ακολουθίας.

Τα επαναληπτικά δίκτυα, είτε μονόδρομα είτε αμφίδρομα, εντοπίζουν μοτίβα μέσα σε μια ακολουθία και τα αποθηκεύουν σε διανύσματα κατάστασης ή τα επιστρέφουν ως έξοδο. Όπως με τα συνελικτικά δίκτυα, μπορούμε να κατασκευάσουμε ένα ακόμη επαναληπτικό επίπεδο μετά το πρώτο για να εντοπίσουμε επίπεδα υψηλότερου επιπέδου μοτίβα, που σχηματίζονται από μοτίβα χαμηλότερου επιπέδου που εξάγει το πρώτο επίπεδο. Αυτό μας οδηγεί στην έννοια του **πολυεπίπεδου RNN**, που αποτελείται από δύο ή περισσότερα επαναληπτικά δίκτυα, όπου η έξοδος του προηγούμενου επιπέδου περνά στην επόμενη στρώση ως είσοδος.

![Εικόνα που δείχνει ένα Πολυεπίπεδο long-short-term-memory RNN](../../../../../translated_images/el/multi-layer-lstm.dd975e29bb2a59fe.webp)

*Εικόνα από [αυτή την υπέροχη ανάρτηση](https://towardsdatascience.com/from-a-lstm-cell-to-a-multilayer-lstm-network-with-pytorch-2899eb5696f3) του Fernando López.*

Το Keras κάνει την κατασκευή αυτών των δικτύων εύκολη υπόθεση, επειδή απλά χρειάζεται να προσθέσουμε περισσότερα επαναληπτικά επίπεδα στο μοντέλο. Για όλα τα επίπεδα εκτός του τελευταίου, πρέπει να ορίσουμε την παράμετρο `return_sequences=True`, γιατί χρειαζόμαστε το επίπεδο να επιστρέφει όλες τις ενδιάμεσες καταστάσεις και όχι μόνο την τελική κατάσταση του επαναληπτικού υπολογισμού.

Ας φτιάξουμε ένα αμφίδρομο LSTM δύο επιπέδων για το πρόβλημα ταξινόμησης μας.

> **Σημείωση** αυτός ο κώδικας πάλι παίρνει αρκετό χρόνο για να ολοκληρωθεί, αλλά μας δίνει την υψηλότερη ακρίβεια που έχουμε δει μέχρι τώρα. Οπότε ίσως αξίζει να περιμένουμε και να δούμε το αποτέλεσμα.


In [9]:
model = keras.models.Sequential([
    vectorizer,
    keras.layers.Embedding(vocab_size, 128, mask_zero=True),
    keras.layers.Bidirectional(keras.layers.LSTM(64,return_sequences=True)),
    keras.layers.Bidirectional(keras.layers.LSTM(64)),    
    keras.layers.Dense(4,activation='softmax')
])

model.compile(loss='sparse_categorical_crossentropy',metrics=['acc'], optimizer='adam')
model.fit(ds_train.map(tupelize).batch(batch_size),
          validation_data=ds_test.map(tupelize).batch(batch_size))

5045/7500 [===================>..........] - ETA: 2:33 - loss: 0.3709 - acc: 0.8706

## RNNs για άλλες εργασίες

Μέχρι τώρα, έχουμε επικεντρωθεί στη χρήση RNNs για την ταξινόμηση ακολουθιών κειμένου. Αλλά μπορούν να χειριστούν πολλές άλλες εργασίες, όπως η δημιουργία κειμένου και η μηχανική μετάφραση &mdash; θα εξετάσουμε αυτές τις εργασίες στην επόμενη ενότητα.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Αποποίηση ευθυνών**:
Αυτό το έγγραφο έχει μεταφραστεί χρησιμοποιώντας την υπηρεσία μετάφρασης με τεχνητή νοημοσύνη [Co-op Translator](https://github.com/Azure/co-op-translator). Ενώ επιδιώκουμε την ακρίβεια, παρακαλούμε να έχετε υπόψη ότι οι αυτοματοποιημένες μεταφράσεις ενδέχεται να περιέχουν λάθη ή ανακρίβειες. Το πρωτότυπο έγγραφο στη μητρική του γλώσσα πρέπει να θεωρείται η αυθεντική πηγή. Για κρίσιμες πληροφορίες, συνιστάται επαγγελματική ανθρώπινη μετάφραση. Δεν φέρουμε ευθύνη για τυχόν παρεξηγήσεις ή λανθασμένες ερμηνείες που προκύπτουν από τη χρήση αυτής της μετάφρασης.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
